In [ ]:
def are_providers_set(providers):
    for _, value in providers.items():
        if not value["endpoint"] or not value["api_key"]:
            return False
    return True

In [ ]:
def get_provider_credentials(providers, provider_name):
    provider = providers.get(provider_name, None)
    false = False, "", ""
    if provider:
        endpoint = provider.get("endpoint", "")
        api_key = provider.get("api_key", "")
        if endpoint == "" or api_key == "":
            return false
        return True, endpoint, api_key
    return false

In [ ]:
def prompt_agent(client, model, message, instructions=None, limit_max_tokens=False):
    messages = []

    if not message:
        raise ValueError("Messages not found")

    messages.append(
        {
            "role": "user",
            "content": message,
        },
    )
    
    if instructions:
        messages.append(
            {
                "role": "system",
                "content": instructions,
            }
        )

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        max_completion_tokens=2500 if limit_max_tokens else None,
    )
    return response

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

load_dotenv()

# AI Providers
providers = {
    "foundry": {
        "endpoint": os.getenv("AZURE_FOUNDRY_ENDPOINT"),
        "api_key": os.getenv("AZURE_FOUNDRY_API_KEY"),
    },
    "groq": {
        "endpoint": os.getenv("GROQ_ENDPOINT"),
        "api_key": os.getenv("GROQ_API_KEY"),
    },
    "openrouter": {
        "endpoint": os.getenv("OPENROUTER_ENDPOINT"),
        "api_key": os.getenv("OPENROUTER_API_KEY"),
    },
}

try:
    ok, endpoint, api_key = get_provider_credentials(providers, "foundry")

    if not ok:
        raise Exception("Provider not found")

    foundry = OpenAI(base_url=endpoint, api_key=api_key)

    # Asking the LLM to come up with a question for a user
    response = prompt_agent(
        foundry,
        "gpt-5-nano",
        message="""Please come up with a challenging, nuanced question with a succinct answer,
                    that I can ask a number of LLMs to evaluate their intelligence.
                    Not a mathematical puzzle, but more of a thought-provoking question that requires intelligent insight.
                    Consider only on of the three areas: philosophy, psychology and medicine.
                    Include in your question that the answer must be short.
                    Provide only the question in plain text and nothing more.
                    Do not include the area of the question or any other extra information""",
        limit_max_tokens=True,
    )
    question = response.choices[0].message.content
    print(f"The question is: {question}")

    # Categorize user's question
    ok, endpoint, api_key = get_provider_credentials(providers, "groq")

    if not ok:
        raise Exception("Provider not found")
    
    groq = OpenAI(base_url=endpoint, api_key=api_key)

    response = prompt_agent(
        groq,
        "openai/gpt-oss-120b",
        message=f"""The question is: \n {question}""",
        instructions="""You are an assistant that categorizes the user's question in on of three different areas: philosophy, psychology and medicine. 
                            Read the user prompt and categorize the user input. 
                            Return only one word in lowercase that indicates the category.""",
        limit_max_tokens=True,
    )
    category = response.choices[0].message.content
    print(f"The category of the question is: {category}")

    # Routing the user's question
    if category == "philosophy":
        ok, endpoint, api_key = get_provider_credentials(providers, "openrouter")

        if not ok:
            raise Exception("Provider not found")

        openrouter = OpenAI(base_url=endpoint, api_key=api_key)

        response = prompt_agent(
            openrouter,
            "openrouter/free",
            message=f"""Provide a clear and short answer from the user related to {category}:
                        {question}
                        Provide your answer in markdown format only.""",
            instructions=f"""You are an assistant that knows everything about {category}.
                            You cannot answer any other type of question.
                            If the question is not {category} related, answer the following:
                            I am sorry, I do not have an answer for that, I just know everything about {category}!.""",
            limit_max_tokens=True,
        )
        print(f"openrouter response:")
        display(Markdown(response.choices[0].message.content))

    if category == "psychology":
        ok, endpoint, api_key = get_provider_credentials(providers, "groq")

        if not ok:
            raise Exception("Provider not found")

        groq = OpenAI(base_url=endpoint, api_key=api_key)

        response = prompt_agent(
            groq,
            "openai/gpt-oss-120b",
            message=f"""Provide a clear and short answer from the user related to {category}:
                        {question}
                        Provide your answer in markdown format only.""",
            instructions=f"""You are an assistant that knows everything about {category}.
                    You cannot answer any other type of question.
                    If the question is not {category} related, answer the following:
                    I am sorry, I do not have an answer for that, I just know everything about {category}!.""",
        )
        print(f"qroq response:")
        display(Markdown(response.choices[0].message.content))

    if category == "medicine":
        ok, endpoint, api_key = get_provider_credentials(providers, "foundry")

        if not ok:
            raise Exception("Provider not found")

        foundry = OpenAI(base_url=endpoint, api_key=api_key)

        response = prompt_agent(
            foundry,
            "DeepSeek-V4-Flash",
            message=f"""Provide a clear and short answer from the user related to {category}:
                        {question}
                        Provide your answer in markdown format only.""",
            instructions=f"""You are an assistant that knows everything about {category}.
                    You cannot answer any other type of question.
                    If the question is not {category} related, answer the following:
                    I am sorry, I do not have an answer for that, I just know everything about {category}!.""",
        )
        print(f"foundry response")  
        display(Markdown(response.choices[0].message.content))

except Exception as ex:
    print(f"An error has occurred:\n {ex}")